In [1]:

#### Calculated climos from history output


In [30]:
import xarray as xr
import numpy as np
import pandas as pd
#import xcdat as xcd

import glob as glob
import os as os
import re as re
import cftime as cft

In [31]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [32]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",   
    cores=4,
    memory="32GB",
    processes=4,      # one process per core (safe default)
    local_directory="/glade/derecho/scratch/rneale/dask-temp"  # <--- custom temp dir
)

cluster.scale(jobs=4)
client = Client(cluster)

client

## Setup Run Information

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45915 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /node/crhtc50.hpc.ucar.edu/45025/proxy/45915/status,
Dashboard: /node/crhtc50.hpc.ucar.edu/45025/proxy/45915/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.100:33959,Workers: 0
Dashboard: /node/crhtc50.hpc.ucar.edu/45025/proxy/45915/status,Total threads: 0
Started: Just now,Total memory: 0 B


Task exception was never retrieved
future: <Task finished name='Task-112887' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:2278> exception=AllExit()>
Traceback (most recent call last):
  File "/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py", line 2287, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finished name='Task-112793' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:2278> exception=AllExit()>
Traceback (most recent call last):
  File "/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py", line 2287, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finished name='Task-112882' coro=<Client._gather.<loc

In [33]:
''' Case Details '''

#run_name = 'f.cam6_3_161.FLTHIST_ne30.ke.001'

#run_name = 'f.e30.FLTHIST.CAM7.L48.001a'
run_name = 'f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1'

#dir0_in = '/glade/derecho/scratch/rneale/archive/'
dir0_in = '/glade/campaign/acom/acom-climate/mijeong/archive/'
#dir0_in = '/glade/derecho/scratch/hannay/archive/'
#dir0_in = '/glade/derecho/scratch/gmarques/archive/'
#dir0_in = '/glade/derecho/scratch/aherring/archive/'


dir0_out = '/glade/derecho/scratch/rneale/archive/'

hist_pref = '.h0a.'
years = [1981,1989]
lt_dim = True   # Add in the single value time dimention for compatibility with some of my other ncl/python scripts


########################################




dir_in = dir0_in+run_name+'/atm/hist/'
dir_out = dir0_out+run_name+'/climo/'

syears = str(years)

# Create dir if needed.

if not os.path.exists(dir_out):
    os.makedirs(dir_out)
    print(f"Directory created: {dir_out}")
else:
    print(f"Directory already exists: {dir_out}")



# Sort files 

files_unsorted = os.path.join(dir_in, run_name+'*'+hist_pref+'*.nc')


# List all files matching the pattern
print(files_unsorted)
files_sorted = sorted(glob.glob(files_unsorted))
print('-List of all h0* files')
print(files_sorted[0])
print(files_sorted[-1])

print()
print('-List of all files for requested years - '+syears[0]+ ' to '+ syears[1])

# Subset files that are withing the year range of interest.

pattern = r'\b\d{4}\b'  # Matches exactly 4-digit numbers as whole words
files_in = []

for ff in files_sorted:
    # Find all 4-digit numbers in the string
    match = re.search(pattern, ff)
    if match:
        year = int(match.group())  # Convert to an integer (removes leading zeros)
        if years[0] <= year <= years[1]:
            files_in.append(ff)  # Add the string to the result if year is valid


#


#print(int(re.search(r'\d{4}', 'file_xx_1999').group()))

#files_in = [f for f in files_sorted if years[0] <= int(re.search(r'\d{4}', f).group()) <= years[1]]


print(files_in[0])
print(files_in[-1])

# File number check
print()
if len(files_in) != 12*(years[1]-years[0]+1): 
    print ('INCORRECT NUMBER OF FILES FOR YEARS REQUESTED -- ',years[0],' to ',years[1]) 
    sys.exit
else:
    print ('CORRECT NUMBER OF FILES FOR YEARS REQUESTED -- ',years[0],' to ',years[1]) 


Directory already exists: /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/
/glade/campaign/acom/acom-climate/mijeong/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/atm/hist/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1*.h0a.*.nc
-List of all h0* files
/glade/campaign/acom/acom-climate/mijeong/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/atm/hist/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1.cam.h0a.1980-01.nc
/glade/campaign/acom/acom-climate/mijeong/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/atm/hist/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1.cam.h0a.1993-12.nc

-List of all files for requested years - [ to 1
/glade/campaign/acom/acom-climate/mijeong/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/atm/hist/f.e30.FHISTC_WAt1ma.ne30pg3_m

In [34]:
'''
    Read in data (lazy)
'''

ds_hist = xr.open_mfdataset(files_in, parallel=True, chunks={"time": 60})

KeyboardInterrupt: 

In [ ]:

ds_hist.time

In [21]:
start_time = cft.DatetimeNoLeap(years[0], 1, 1)
end_time = cft.DatetimeNoLeap(years[1], 12, 31)

ds_hist = ds_hist.sel(time=slice(start_time, end_time))

In [22]:
'''
    Drop variables not needed and the string ones which cannot be grouped and averaged
'''

drop_vars_xtime = ['date_written','time_written','trop_cld_lev'] # Drop vars that are not time dimensioned, and add back in later if needed.

drop_vars_aer =['bc_c1','bc_c4','dst_c1','dst_c2','dst_c3','ncl_c1','ncl_c2','ncl_c3','num_c1','num_c2','num_c3','num_c4','pom_c1','pom_c4','so4_c1','so4_c2','so4_c3','soa_c1','soa_c2','bc_a1','bc_a4','dst_a1','dst_a2','dst_a3','ncl_a1','ncl_a2','ncl_a3','num_a1','num_a2','num_a3','so4_a1','so4_a2','so4_a3','soa_a1','soa_a2']

# Drop other unwanted 3D vars.
drop_vars_3d = ['ADRAIN','ADSNOW','ANSNOW','AWNI','AREI','AREL','AWNC','CCN3','CFC11','CFC12','CH4','CO2','DMS','GRAUQM','H2O2','H2SO4','N2O','NUMGRA','SNOWQM','SO2','SOAE','SOAG']

drop_vars_aer = [var for var in drop_vars_aer if var in ds_hist.data_vars]
ds_vars = ds_hist.drop_vars(drop_vars_aer)

drop_vars_3d = [var for var in drop_vars_3d if var in ds_hist.data_vars]
ds_vars = ds_vars.drop_vars(drop_vars_3d)

drop_vars_xtime = [var for var in drop_vars_xtime if var in ds_hist.data_vars]
ds_vars = ds_vars.drop_vars(drop_vars_xtime)

ds_vars.attrs["History Directory"] = dir_in
ds_vars.attrs["Start Year"] = str(years[0])
ds_vars.attrs["End Year"] = str(years[1])
    
# Add the trailing single value time dimension?
# If yes set up the time dimenstion and append later.

#if lt_dim:
    
#    time0 = pd.Timestamp(years[0]+"-01-01")

# Add attributes to the time coordinate
#    time0.attrs["long_name"] = "Time"
#    time0.attrs["standard_name"] = "time"
#    time0.attrs["description"] = "Time coordinate for the data"   ds_vars.coords["time"].attrs["calendar"] = "gregorian"


ds_vars

<xarray.Dataset> Size: 54GB
Dimensions:       (time: 108, lat: 257, lev: 135, ilev: 136, nbnd: 2, lon: 512,
                   trop_pref: 100, trop_prefi: 101, trop_cld_lev: 100)
Coordinates:
  * lat           (lat) float64 2kB -90.0 -89.3 -88.59 ... 88.59 89.3 90.0
  * lon           (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
  * lev           (lev) float64 1kB 6.14e-06 1.567e-05 2.94e-05 ... 991.2 997.5
  * ilev          (ilev) float64 1kB 2.043e-06 1.024e-05 ... 995.1 1e+03
  * trop_pref     (trop_pref) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
  * trop_prefi    (trop_prefi) float64 808B 0.9833 1.142 1.317 ... 995.1 1e+03
  * trop_cld_lev  (trop_cld_lev) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
  * time          (time) object 864B 1981-01-16 12:00:00 ... 1989-12-16 12:00:00
Dimensions without coordinates: nbnd
Data variables: (12/29)
    w             (time, lat) float64 222kB dask.array<chunksize=(1, 257), meta=np.ndarray>
    hyam          (time, lev) float64 117kB dask.array<chunksize=(1, 135), meta=np.ndarray>
    hybm          (time, lev) float64 117kB dask.array<chunksize=(1, 135), meta=np.ndarray>
    hyai          (time, ilev) float64 118kB dask.array<chunksize=(1, 136), meta=np.ndarray>
    hybi          (time, ilev) float64 118kB dask.array<chunksize=(1, 136), meta=np.ndarray>
    date          (time) int32 432B dask.array<chunksize=(1,), meta=np.ndarray>
    ...            ...
    Q             (time, lev, lat, lon) float32 8GB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    T             (time, lev, lat, lon) float32 8GB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    U             (time, lev, lat, lon) float32 8GB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    V             (time, lev, lat, lon) float32 8GB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    ZMDT          (time, lev, lat, lon) float32 8GB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    CMFMC_DP      (time, ilev, lat, lon) float32 8GB dask.array<chunksize=(1, 136, 257, 512), meta=np.ndarray>
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_bere...
    logname:            mijeong
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/campaign/acom/acom-climate/mijeong/archive/f.e...
    Start Year:         1981
    End Year:           1989

In [23]:
'''
 Calculate monthly climo - persist to worker memory once; annual and seasonal derive from this.
'''
from dask.distributed import wait

ds_cmonth_base = ds_vars.groupby("time.month").mean(keep_attrs=True)

print('Computing monthly climatology (single full pass over data)...')
ds_cmonth_base = ds_cmonth_base.persist()  # stays on workers, never pulled to kernel
wait(ds_cmonth_base)                        # block until workers finish
print('-Done.')

# Month names for output file.
mname_file = ["%02d" % x for x in ds_cmonth_base.month]

# Add the trailing single value time dimension for compatibility.
ds_cmonth = ds_cmonth_base.expand_dims(time=[ds_vars.time.values[0]])
ds_cmonth.coords["time"].attrs = ds_vars.coords["time"].attrs
ds_cmonth

Computing monthly climatology (single full pass over data)...
-Done.


<xarray.Dataset> Size: 6GB
Dimensions:       (time: 1, month: 12, lat: 257, lev: 135, ilev: 136, lon: 512,
                   trop_pref: 100, trop_prefi: 101, trop_cld_lev: 100)
Coordinates:
  * time          (time) object 8B 1981-01-16 12:00:00
  * lat           (lat) float64 2kB -90.0 -89.3 -88.59 ... 88.59 89.3 90.0
  * lon           (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
  * lev           (lev) float64 1kB 6.14e-06 1.567e-05 2.94e-05 ... 991.2 997.5
  * ilev          (ilev) float64 1kB 2.043e-06 1.024e-05 ... 995.1 1e+03
  * trop_pref     (trop_pref) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
  * trop_prefi    (trop_prefi) float64 808B 0.9833 1.142 1.317 ... 995.1 1e+03
  * trop_cld_lev  (trop_cld_lev) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
  * month         (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Data variables: (12/28)
    w             (time, month, lat) float64 25kB dask.array<chunksize=(1, 1, 257), meta=np.ndarray>
    hyam          (time, month, lev) float64 13kB dask.array<chunksize=(1, 1, 135), meta=np.ndarray>
    hybm          (time, month, lev) float64 13kB dask.array<chunksize=(1, 1, 135), meta=np.ndarray>
    hyai          (time, month, ilev) float64 13kB dask.array<chunksize=(1, 1, 136), meta=np.ndarray>
    hybi          (time, month, ilev) float64 13kB dask.array<chunksize=(1, 1, 136), meta=np.ndarray>
    date          (time, month) float64 96B dask.array<chunksize=(1, 1), meta=np.ndarray>
    ...            ...
    Q             (time, month, lev, lat, lon) float32 853MB dask.array<chunksize=(1, 1, 135, 257, 512), meta=np.ndarray>
    T             (time, month, lev, lat, lon) float32 853MB dask.array<chunksize=(1, 1, 135, 257, 512), meta=np.ndarray>
    U             (time, month, lev, lat, lon) float32 853MB dask.array<chunksize=(1, 1, 135, 257, 512), meta=np.ndarray>
    V             (time, month, lev, lat, lon) float32 853MB dask.array<chunksize=(1, 1, 135, 257, 512), meta=np.ndarray>
    ZMDT          (time, month, lev, lat, lon) float32 853MB dask.array<chunksize=(1, 1, 135, 257, 512), meta=np.ndarray>
    CMFMC_DP      (time, month, ilev, lat, lon) float32 859MB dask.array<chunksize=(1, 1, 136, 257, 512), meta=np.ndarray>
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_bere...
    logname:            mijeong
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/campaign/acom/acom-climate/mijeong/archive/f.e...
    Start Year:         1981
    End Year:           1989

In [24]:
print('-Calculate annual climatology (derived from monthly climo, no extra I/O)...')

ds_cyear = ds_cmonth_base.mean(dim='month', keep_attrs=True)
ds_cyear

-Calculate annual climatology (derived from monthly climo, no extra I/O)...


<xarray.Dataset> Size: 503MB
Dimensions:       (lat: 257, lev: 135, ilev: 136, lon: 512, trop_pref: 100,
                   trop_prefi: 101, trop_cld_lev: 100)
Coordinates:
  * lat           (lat) float64 2kB -90.0 -89.3 -88.59 ... 88.59 89.3 90.0
  * lon           (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
  * lev           (lev) float64 1kB 6.14e-06 1.567e-05 2.94e-05 ... 991.2 997.5
  * ilev          (ilev) float64 1kB 2.043e-06 1.024e-05 ... 995.1 1e+03
  * trop_pref     (trop_pref) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
  * trop_prefi    (trop_prefi) float64 808B 0.9833 1.142 1.317 ... 995.1 1e+03
  * trop_cld_lev  (trop_cld_lev) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
Data variables: (12/28)
    w             (lat) float64 2kB dask.array<chunksize=(257,), meta=np.ndarray>
    hyam          (lev) float64 1kB dask.array<chunksize=(135,), meta=np.ndarray>
    hybm          (lev) float64 1kB dask.array<chunksize=(135,), meta=np.ndarray>
    hyai          (ilev) float64 1kB dask.array<chunksize=(136,), meta=np.ndarray>
    hybi          (ilev) float64 1kB dask.array<chunksize=(136,), meta=np.ndarray>
    date          float64 8B dask.array<chunksize=(), meta=np.ndarray>
    ...            ...
    Q             (lev, lat, lon) float32 71MB dask.array<chunksize=(135, 257, 512), meta=np.ndarray>
    T             (lev, lat, lon) float32 71MB dask.array<chunksize=(135, 257, 512), meta=np.ndarray>
    U             (lev, lat, lon) float32 71MB dask.array<chunksize=(135, 257, 512), meta=np.ndarray>
    V             (lev, lat, lon) float32 71MB dask.array<chunksize=(135, 257, 512), meta=np.ndarray>
    ZMDT          (lev, lat, lon) float32 71MB dask.array<chunksize=(135, 257, 512), meta=np.ndarray>
    CMFMC_DP      (ilev, lat, lon) float32 72MB dask.array<chunksize=(136, 257, 512), meta=np.ndarray>
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_bere...
    logname:            mijeong
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/campaign/acom/acom-climate/mijeong/archive/f.e...
    Start Year:         1981
    End Year:           1989

In [25]:
print('-Calculate weighted seasonal climatology (derived from monthly climo, no extra I/O)...')

_days = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
_season_months = {'DJF': [12,1,2], 'MAM': [3,4,5], 'JJA': [6,7,8], 'SON': [9,10,11]}

seas_list = []
for sname, months in _season_months.items():
    print(f'  {sname}...')
    days  = np.array([_days[m] for m in months], dtype=float)
    weights = days / days.sum()

    # Direct weighted sum — avoids building a large task graph via xr.concat(_m)
    weighted = None
    for w, m in zip(weights, months):
        term = float(w) * ds_cmonth_base.sel(month=m)
        weighted = term if weighted is None else weighted + term

    # Persist each season on workers before building the next one.
    # This simplifies the graph so the final xr.concat is cheap.
    weighted = weighted.astype(np.float32).persist()
    wait(weighted)
    seas_list.append(weighted.expand_dims(season=[sname]))

ds_wcseas = xr.concat(seas_list, dim='season')
ds_wcseas

-Calculate weighted seasonal climatology (derived from monthly climo, no extra I/O)...
  DJF...
  MAM...
  JJA...
  SON...


<xarray.Dataset> Size: 2GB
Dimensions:       (season: 4, lat: 257, lev: 135, ilev: 136, lon: 512,
                   trop_pref: 100, trop_prefi: 101, trop_cld_lev: 100)
Coordinates:
  * season        (season) object 32B 'DJF' 'MAM' 'JJA' 'SON'
  * lat           (lat) float64 2kB -90.0 -89.3 -88.59 ... 88.59 89.3 90.0
  * lon           (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
  * lev           (lev) float64 1kB 6.14e-06 1.567e-05 2.94e-05 ... 991.2 997.5
  * ilev          (ilev) float64 1kB 2.043e-06 1.024e-05 ... 995.1 1e+03
  * trop_pref     (trop_pref) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
  * trop_prefi    (trop_prefi) float64 808B 0.9833 1.142 1.317 ... 995.1 1e+03
  * trop_cld_lev  (trop_cld_lev) float64 800B 1.062 1.229 1.415 ... 991.2 997.5
    month         (season) int64 32B 2 5 8 11
Data variables: (12/28)
    w             (season, lat) float32 4kB dask.array<chunksize=(1, 257), meta=np.ndarray>
    hyam          (season, lev) float32 2kB dask.array<chunksize=(1, 135), meta=np.ndarray>
    hybm          (season, lev) float32 2kB dask.array<chunksize=(1, 135), meta=np.ndarray>
    hyai          (season, ilev) float32 2kB dask.array<chunksize=(1, 136), meta=np.ndarray>
    hybi          (season, ilev) float32 2kB dask.array<chunksize=(1, 136), meta=np.ndarray>
    date          (season) float32 16B dask.array<chunksize=(1,), meta=np.ndarray>
    ...            ...
    Q             (season, lev, lat, lon) float32 284MB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    T             (season, lev, lat, lon) float32 284MB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    U             (season, lev, lat, lon) float32 284MB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    V             (season, lev, lat, lon) float32 284MB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    ZMDT          (season, lev, lat, lon) float32 284MB dask.array<chunksize=(1, 135, 257, 512), meta=np.ndarray>
    CMFMC_DP      (season, ilev, lat, lon) float32 286MB dask.array<chunksize=(1, 136, 257, 512), meta=np.ndarray>

In [26]:
# Need to copy the attributes due to the array multiplcation wiping them out.

# Global 

ds_wcseas.attrs = ds_vars.attrs 

# Variable

for var in ds_wcseas.data_vars:
    ds_wcseas[var].attrs = ds_vars[var].attrs



In [27]:
'''
    Writing out climo. files
'''
import dask

if not os.path.exists(dir_out): os.makedirs(dir_out)

print('-Writing monthly climatologies in parallel...')

delayed_writes = []
for imm, mname in enumerate(ds_cmonth.month.values):
    fout_mon = dir_out + run_name + '_' + mname_file[imm] + '_climo.nc'
    print(mname_file[imm], ' -- Queuing - ', fout_mon)
    delayed_writes.append(ds_cmonth.sel(month=mname).to_netcdf(fout_mon, mode="w", compute=False))

dask.compute(*delayed_writes)
print('-Done writing monthly files...')

# Writing out seasonal climatologies

-Writing monthly climatologies in parallel...
01  -- Queuing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_01_climo.nc
02  -- Queuing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_02_climo.nc
03  -- Queuing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_03_climo.nc
04  -- Queuing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_04_climo.nc
05  -- Queuing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15

In [28]:
for sname in ds_wcseas.season.values:
    fout_seas = dir_out+run_name+'_'+sname+'_climo.nc'
    print(sname,'-- Writing - ',fout_seas)
    ds_wcseas.sel(season=sname).drop_vars('season').to_netcdf(fout_seas, mode="w")
    print('-Done...')

DJF -- Writing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_DJF_climo.nc
-Done...
MAM -- Writing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_MAM_climo.nc
-Done...
JJA -- Writing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_JJA_climo.nc
-Done...
SON -- Writing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_SON_climo.nc
-Done...


In [29]:
# Wrting out annual climatology
fout_ann = dir_out+run_name+'_ANN_climo.nc'

print('-Writing - ',fout_ann)
ds_cyear.to_netcdf(fout_ann, mode="w")
print('-Done...')

print('---- COMPLETE ----')

-Writing -  /glade/derecho/scratch/rneale/archive/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1/climo/f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1_ANN_climo.nc
-Done...
---- COMPLETE ----
